# DMRGT Phase 1 — verification notebook

This notebook is for **manual hand-checks** of the local pipeline.
Run the cells in order after `python data.py --download` and
`python data.py --targets` have both completed.

The single most important cell is **§2 — hand-check table for one date**,
which prints, for each ticker, the close at `t`, the close at `t+1`,
the raw return, the universe mean, and the resulting excess return.
You should be able to read across a row and confirm the arithmetic by
eye.


In [1]:
import sys
from pathlib import Path
# Make the project root importable. Walk up from CWD until we find
# configs.yaml — works whether the kernel was started from the project
# root, from notebooks/, or from anywhere else.
def _find_project_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "configs.yaml").exists():
            return p
    return Path.cwd().resolve()

project_root = _find_project_root()
sys.path.insert(0, str(project_root))
print(f"project root: {project_root}")

import pandas as pd
import data, paths

# Load caches
prices = pd.read_parquet(paths.PRICES)
targets = pd.read_parquet(paths.TARGETS)
print(f"prices:  {len(prices):,} rows, {prices['ticker'].nunique()} tickers, "
      f"{prices['date'].min().date()} → {prices['date'].max().date()}")
print(f"targets: {len(targets):,} rows, {targets['ticker'].nunique()} tickers, "
      f"{targets['date'].min().date()} → {targets['date'].max().date()}")


project root: /Users/harishsivakumar/Documents/VIT-Project-1/DMRGT


prices:  55,330 rows, 22 tickers, 2015-01-02 → 2024-12-30
targets: 50,280 rows, 20 tickers, 2015-01-02 → 2024-12-27


## §1 — Cross-sectional sum (mandatory §4 check)

For every date, excess returns across all 20 stocks must sum to ≈ 0.


In [2]:
sums = targets.groupby("date")["excess_return"].sum().abs()
print(f"dates checked:        {len(sums)}")
print(f"max |sum|:            {sums.max():.3e}")
print(f"mean |sum|:           {sums.mean():.3e}")
print(f"all < 1e-10?          {(sums < 1e-10).all()}")


dates checked:        2514
max |sum|:            2.220e-16
mean |sum|:           5.121e-18
all < 1e-10?          True


## §2 — Hand-check table for one date

For one chosen date `t`, this table shows every stock's `close[t]`,
`close[t+1]`, the raw return realised on `t+1`, the equal-weight
universe mean, and the resulting excess return. The target value in
the last column must equal `return[t+1] - mean[t+1]` to within
floating-point precision.


In [3]:
cfg = data.load_config(str(project_root / "configs.yaml"))
universe = list(cfg["universe"]["tickers"])

all_dates = sorted(prices["date"].unique())
if pd.Timestamp("2018-01-02") in pd.to_datetime(all_dates):
    chosen = pd.Timestamp("2018-01-02")
else:
    chosen = pd.Timestamp(all_dates[100])
next_date_idx = all_dates.index(chosen) + 1
next_date = pd.Timestamp(all_dates[next_date_idx])
print(f"chosen t   = {chosen.date()}  (label describes what happened on this day)")
print(f"chosen t+1 = {next_date.date()}  (the day whose return is the label)")
print()

px = prices.pivot(index="date", columns="ticker", values="close").sort_index()
close_t  = px.loc[chosen]
close_t1 = px.loc[next_date]
raw_ret  = (close_t1 / close_t) - 1
mean_t1  = raw_ret[universe].mean()
excess   = raw_ret[universe] - mean_t1

table = pd.DataFrame({
    "close[t]"   : close_t[universe].values,
    "close[t+1]" : close_t1[universe].values,
    "raw_ret[t+1]": raw_ret[universe].values,
    "mean[t+1]"  : [mean_t1] * len(universe),
    "excess[t+1]": excess.values,
}, index=universe)
table.index.name = "ticker"

with pd.option_context("display.float_format", "{:,.6f}".format, "display.width", 140):
    print(table.to_string())

target_row = targets[targets["date"] == chosen].set_index("ticker")["excess_return"]
diff = (table["excess[t+1]"] - target_row).abs().max()
print()
print(f"max |table excess - cached target| = {diff:.3e}   (should be ≈ 0)")


chosen t   = 2018-01-02  (label describes what happened on this day)
chosen t+1 = 2018-01-03  (the day whose return is the label)

         close[t]  close[t+1]  raw_ret[t+1]  mean[t+1]  excess[t+1]
ticker                                                             
AAPL    40.267067   40.260071     -0.000174   0.002509    -0.002683
MSFT    78.699898   79.066154      0.004654   0.002509     0.002145
JPM     85.516037   85.603180      0.001019   0.002509    -0.001490
BRK-B  197.220001  199.789993      0.013031   0.002509     0.010522
GS     212.209000  210.233597     -0.009309   0.002509    -0.011818
JNJ    110.130821  111.182861      0.009553   0.002509     0.007043
PFE     23.040384   23.211086      0.007409   0.002509     0.004900
UNH    191.993896  194.007935      0.010490   0.002509     0.007981
XOM     58.185493   59.328262      0.019640   0.002509     0.017131
CVX     88.640480   89.286644      0.007290   0.002509     0.004780
AMZN    59.450500   60.209999      0.012775   0.00250

## §3 — Pick your own (date, ticker) and verify by hand

Change `MY_DATE` and `MY_TICKER` below to any (date, ticker) pair and
the cell will print every intermediate value. The target value at
`(MY_DATE, MY_TICKER)` should equal the displayed `excess = ...`.


In [4]:
MY_DATE   = pd.Timestamp("2018-01-02")
MY_TICKER = "AAPL"

all_dates = sorted(prices["date"].unique())
next_d = pd.Timestamp(all_dates[all_dates.index(MY_DATE) + 1])

px = prices.pivot(index="date", columns="ticker", values="close").sort_index()
cfg = data.load_config(str(project_root / "configs.yaml"))
universe = list(cfg["universe"]["tickers"])

all_rets_t1 = (px.loc[next_d] / px.shift(1).loc[next_d]) - 1
mean_t1     = float(all_rets_t1[universe].mean())
excess      = float(all_rets_t1[MY_TICKER] - mean_t1)
cached      = float(targets[(targets["date"] == MY_DATE) &
                            (targets["ticker"] == MY_TICKER)]["excess_return"].iloc[0])

print(f"date          = {MY_DATE.date()}")
print(f"ticker        = {MY_TICKER}")
print(f"close[t]      = {float(px.loc[MY_DATE, MY_TICKER]):.6f}")
print(f"close[t+1]    = {float(px.loc[next_d, MY_TICKER]):.6f}   (t+1 = {next_d.date()})")
print(f"raw return    = {float(all_rets_t1[MY_TICKER]):.6f}")
print(f"universe mean = {mean_t1:.6f}   (equal-weight across 20 stocks at t+1)")
print(f"excess = ret - mean = {excess:.6f}")
print(f"cached target = {cached:.6f}")
print(f"diff          = {abs(excess - cached):.3e}")


date          = 2018-01-02
ticker        = AAPL
close[t]      = 40.267067
close[t+1]    = 40.260071   (t+1 = 2018-01-03)
raw return    = -0.000174
universe mean = 0.002509   (equal-weight across 20 stocks at t+1)
excess = ret - mean = -0.002683
cached target = -0.002683
diff          = 0.000e+00


## §4 — No-lookahead and structure

The label at the final price date is dropped (we can't know t+1 for
the last day). This cell asserts the structural invariants.


In [5]:
last_price_date  = pd.Timestamp(prices["date"].max())
last_target_date = pd.Timestamp(targets["date"].max())
per_date_counts  = targets.groupby("date").size()

print(f"last price date:    {last_price_date.date()}")
print(f"last target date:   {last_target_date.date()}")
print(f"dropped (last day): {last_price_date.date() != last_target_date.date()}")
print()
print(f"per-date row count (should be exactly 20 on every date):")
print(f"  min:  {per_date_counts.min()}")
print(f"  max:  {per_date_counts.max()}")
print(f"  all 20?  {(per_date_counts == 20).all()}")


last price date:    2024-12-30
last target date:   2024-12-27
dropped (last day): True

per-date row count (should be exactly 20 on every date):
  min:  20
  max:  20
  all 20?  True
